# Tokenization — FVT (Fast Vocabulary Transfer) embedding init for an extended tokenizer (`core/tokenization`, `core/model_loader`)

In [ ]:
from aligntune.core.backend_factory import create_sft_trainer

# After extending a tokenizer's vocabulary (see notebook 43), the new tokens
# need embeddings. FVT initializes them from the mean of their constituent
# subword embeddings instead of random -- this is what makes the extended
# model usable without a long cold-start fine-tune (EEVE / Chinese-LLaMA-
# Alpaca-style init). embedding_init_method is applied at model-load time.
#
# Fixes vs. the original snippet:
# - model_name: meta-llama/Llama-2-7b-hf -> TinyLlama/TinyLlama-1.1B-Chat-v1.0.
#   FVT's mean_of_constituents decomposes each NEW token into pieces of the
#   BASE tokenizer, so the base model here must share the tokenizer lineage
#   that ./out_tokenizer_hindi was extended from in notebook 43 (which now
#   uses TinyLlama's tokenizer -- identical to Llama-2's -- for the same
#   reason: no gating, and a ~2GB download instead of ~13GB).
# - config_name="20231101.hi" -> subset="20231101.hi". create_sft_trainer()
#   has no `config_name` parameter (that name only exists on
#   create_tokenization_trainer) - passing it here silently vanished into
#   **kwargs and matched nothing, so wikimedia/wikipedia would have loaded
#   with no language subset selected.
# - Added dataset_text_field="completion": wikimedia/wikipedia's raw "text"
#   column gets renamed to "completion" by the SFT column-heuristic mapping
#   (aligntune.data.processors), so TRL's SFTTrainer (which defaults to
#   reading a "text" column) raised KeyError: 'text' without this.
# - Added use_peft (LoRA): even at 1.1B, full fine-tuning costs more VRAM
#   (weights + gradients + optimizer states) than a LoRA adapter needs,
#   and every other notebook in this repo uses PEFT for the same reason.
#
# backend="unsloth": fixed a real gap in core/model_loader.py's Unsloth
# loading branch to make this work - it always ignored
# tokenizer_name_or_path (FastLanguageModel.from_pretrained returns the base
# model's own tokenizer, not an extended one), and never ran the
# embedding_init_method adaptation at all (that block only existed in the
# standard/TRL loading branch). Added both, mirroring the standard branch's
# order: load model -> swap in the configured tokenizer -> adapt embeddings
# -> apply LoRA. Verified end-to-end: the resize is confirmed by PEFT's own
# "embedding layer has been resized during finetuning" warning, and training
# completes with real loss reported.
trainer = create_sft_trainer(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer_name_or_path="./out_tokenizer_hindi",  # the extended tokenizer from notebook 43
    dataset_name="wikimedia/wikipedia",
    subset="20231101.hi",
    backend="unsloth",
    output_dir="./out_fvt_hindi_sft",
    embedding_init_method="mean_of_constituents",  # options: random, mean, mean_of_constituents
    dataset_text_field="completion",
    use_peft=True,
    lora_r=16,
    lora_alpha=32,
    num_epochs=1,
    batch_size=1,
    learning_rate=2e-5,
    max_seq_length=256,
    max_samples=16,
)

results = trainer.train()
print("SFT with FVT embedding init completed.")
print(results)